```
# Lab type:  debug
# Course:    NL301 Natural Language Processing with Python
# Lesson:    09 — Embedding Failure Modes
# Task:      Find and fix three bugs in a FAISS vector search pipeline.
```

## Setup and reference code

In [ ]:
!pip install faiss-cpu sentence-transformers numpy --quiet
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing involves text analysis and understanding",
    "computer vision enables machines to interpret visual information",
    "reinforcement learning trains agents through rewards and penalties",
]
query = "neural network architectures for deep learning"

# Reference — correct search
def search_correct(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embs)
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(qemb)
    D, I = index.search(qemb, top_k)
    print("Reference results:")
    for i, d in zip(I[0], D[0]):
        print(f"  {d:.4f}  {corpus[i]}")

search_correct(query, corpus)


---
## Bug 1: Missing L2 normalisation before IndexFlatIP

`IndexFlatIP` computes **dot product**, not cosine similarity. Without normalisation, longer vectors score higher regardless of direction.

In [ ]:
# BUG: normalisation step missing
def search_buggy_1(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    # faiss.normalize_L2(embs)  ← Bug: commented out
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    # faiss.normalize_L2(qemb) ← Bug: commented out
    D, I = index.search(qemb, top_k)
    print("Buggy results (no normalisation):")
    for i, d in zip(I[0], D[0]):
        print(f"  {d:.4f}  {corpus[i]}")

search_buggy_1(query, corpus)


**Explain the bug** — what does `IndexFlatIP` compute without normalisation? Why can this produce incorrect rankings?

*(Write your answer here.)*

In [ ]:
# FIX 1: normalise both corpus and query embeddings before indexing
def search_fixed_1(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embs)                 # fix
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(qemb)                 # fix
    D, I = index.search(qemb, top_k)
    print("Fixed results:")
    for i, d in zip(I[0], D[0]):
        print(f"  {d:.4f}  {corpus[i]}")

search_fixed_1(query, corpus)


---
## Bug 2: L2 distances sorted descending

`IndexFlatL2` returns **L2 distances** — lower is more similar. Sorting descending puts the worst results first.

In [ ]:
# BUG: IndexFlatL2 results sorted descending (worst-first)
def search_buggy_2(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    dim = embs.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    D, I = index.search(qemb, len(corpus))   # get all, then sort manually
    pairs = sorted(zip(D[0], I[0]), reverse=True)  # ← Bug: should be ascending
    print("Buggy results (highest L2 distance first):")
    for d, i in pairs[:top_k]:
        print(f"  L2={d:.4f}  {corpus[i]}")

search_buggy_2(query, corpus)

print()
print("All L2 distances for reference:")
embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
index_l2 = faiss.IndexFlatL2(embs.shape[1])
index_l2.add(embs)
qemb = model.encode([query], convert_to_numpy=True).astype('float32')
D, I = index_l2.search(qemb, len(corpus))
for d, i in zip(D[0], I[0]):
    print(f"  L2={d:.4f}  {corpus[i]}")


**Explain the bug** — what does L2 distance represent? Why must you sort ascending, not descending?

*(Write your answer here.)*

In [ ]:
# FIX 2: sort ascending for L2 distances
def search_fixed_2(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    dim = embs.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    D, I = index.search(qemb, len(corpus))
    pairs = sorted(zip(D[0], I[0]))   # ascending: smallest L2 = most similar
    print("Fixed results (lowest L2 distance first):")
    for d, i in pairs[:top_k]:
        print(f"  L2={d:.4f}  {corpus[i]}")

search_fixed_2(query, corpus)


---
## Bug 3: No deduplication before indexing

A web-scraped corpus contains near-duplicate documents. All top-k results are the same document repeated.

In [ ]:
# BUG: duplicates in corpus not removed before indexing
corpus_with_dupes = [
    "deep learning uses neural networks with many layers",
    "deep learning uses neural networks with many layers",   # exact duplicate
    "Deep learning uses neural networks with many layers.",  # near-duplicate
    "machine learning is a subset of artificial intelligence",
    "natural language processing involves text analysis",
]

def search_buggy_3(query, corpus, top_k=3):
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(qemb)
    D, I = index.search(qemb, top_k)
    print("Buggy results (duplicates not removed):")
    for i, d in zip(I[0], D[0]):
        print(f"  {d:.4f}  {corpus[i]}")

search_buggy_3(query, corpus_with_dupes)


**Explain the bug** — what is the user experience when the top-3 results are the same document? How do hash deduplication and near-duplicate cosine threshold handle different types of duplicates?

*(Write your answer here.)*

In [ ]:
# FIX 3: deduplicate before indexing
def deduplicate(corpus: list[str], sim_threshold: float = 0.98) -> list[str]:
    # Step 1: exact deduplication by hash
    seen = set()
    unique = []
    for doc in corpus:
        h = hash(doc.strip())
        if h not in seen:
            seen.add(h)
            unique.append(doc)
    
    # Step 2: near-duplicate removal by cosine threshold
    embs = model.encode(unique, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embs)
    keep = [0]
    for i in range(1, len(unique)):
        sims = embs[:i] @ embs[i]          # cosine similarity to all kept docs
        if sims.max() < sim_threshold:
            keep.append(i)
    return [unique[i] for i in keep]

deduped = deduplicate(corpus_with_dupes)
print(f"Before: {len(corpus_with_dupes)} docs  After: {len(deduped)} docs")
for doc in deduped:
    print(f"  {doc}")

def search_fixed_3(query, corpus, top_k=3):
    corpus = deduplicate(corpus)
    embs = model.encode(corpus, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    qemb = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(qemb)
    D, I = index.search(qemb, min(top_k, len(corpus)))
    print("Fixed results (after deduplication):")
    for i, d in zip(I[0], D[0]):
        print(f"  {d:.4f}  {corpus[i]}")

search_fixed_3(query, corpus_with_dupes)


---
## Summary

1. Bug 1 fix: ___
2. Bug 2 fix: ___
3. Bug 3 fix: ___